# C15 - MiniTransformer Training on Kaggle (T4 x2)

Notebook Kaggle untuk eksperimen **C15**: language model decoder-only (MiniGPT) di atas
Indonesian News Datasets, subset 1.800 artikel (1.440 train / 159 val / 200 test).

## Kenapa C15 ada

C14 (notebook `kaggle_c14.ipynb`) berhasil menghentikan divergensi val, tetapi analisisnya
menunjukkan tiga batas keras:

1. **Rasio data/parameter 8,8 param per token latih.** 2.556.672 parameter untuk 290.920
   token. 96% parameter ada di embedding + LM head yang tidak di-tie.
2. **Vocabulary 19.192 dengan 8.173 hapax (42,6% tipe).** 1.046.144 parameter dipakai
   token yang muncul sekali; 5,7% token val tidak ada di vocab.
3. **`context_length=64` hanya mencakup ~18% median dokumen (363 token).** Model tidak
   pernah melihat akhir artikel, jadi tidak pernah belajar mengakhiri kalimat
   (`<eos>` hanya 0,29% target window).

## Perubahan C15 terhadap baseline TECH_SPEC

| Aspek | Baseline / C14 | C15 | ADR |
|---|---|---|---|
| Subset | 1.000 artikel (720/80/200) | **1.800 artikel (1.440/159/200)** | ADR-015 |
| `context_length` | 64 | **128** | ADR-015 |
| `min_freq` vocab | 1 (19.192 tipe) | **2 (16.278 tipe)** | ADR-013 |
| Weight tying | tidak | **ya** (`lm_head` = `token_emb`) | ADR-014 |
| `n_layers` | 2 | **4** | ADR-014 |
| `epochs` | 10 | **20** | hiperparameter (TECH_SPEC 1) |
| `d_model`, `n_heads`, `d_ff` | 64 / 4 / 256 | sama | tidak berubah |
| Attention, PE | manual | sama (tidak ada `nn.MultiheadAttention`) | ADR-002 |

Parameter turun dari 2.556.672 menjadi ~1,24 juta, sementara token latih naik dari 290.920
menjadi ~568.000: rasio membaik dari **8,8 menjadi ~2,2 parameter/token**.

## Protokol pengukuran (diwarisi dari C14 Tier 0)

- **Val dihitung per dokumen** (window non-overlap), bukan per window stride-1. Pada C14,
  28.229 window berasal dari 80 dokumen (redundansi ~54x) sehingga CI selebar +/-0,27 nat.
- **CI 95%** dilaporkan (`1.96 * sd / sqrt(n_dokumen)`). Epoch yang selisihnya di bawah CI
  dicetak sebagai `tied_epochs` dan tidak boleh diklaim sebagai "epoch terbaik".
- **`test_ppl`** (200 artikel held-out) adalah angka untuk klaim generalisasi; `val_ppl`
  hanya untuk memantau training.

## Metrik kualitas generasi (jujur, bukan tautologis)

Tiga angka di laporan harus dibaca dengan benar:

| Metrik | Kenapa | Cara baca |
|---|---|---|
| `ends_at_sentence_pre_trim` | Diukur **sebelum** trim. Kalau diukur sesudah, angkanya selalu 100% karena trim sendiri yang memotong di titik | Model benar-benar berhenti di batas kalimat tanpa bantuan `postprocess` |
| `dup_4gram` | `no_repeat_ngram_size=3` **melarang** trigram berulang, jadi `dup_trigram=0` adalah hasil pemaksaan. 4-gram di luar cakupan larangan | 0 berarti tidak ada pengulangan yang dipilih model sendiri |
| `verbatim_span_tokens` | Span terpanjang yang identik dengan teks train | Naik = makin menghafal. Pada C14 nilainya 6,3 rata-rata (maks 12) |

Demo dijalankan pada **10 prompt held-out** (5 val + 5 test), bukan prompt dari train.

## Decoding (diwarisi dari C14 Tier 1)

- Greedy + `no_repeat_ngram_size=3` (dup-trigram 4,67 -> 0,00 per 60 token pada C14).
- Sampling memakai temperature yang **dipilih dengan batasan koherensi**, bukan temperature
  optimal-NLL. Pada model kecil, NLL terus turun sampai T=2,0 padahal outputnya sudah acak.
- `<unk>` dibanned saat inference; teks dirapikan `postprocess.py` (detokenisasi, kapitalisasi,
  potong di batas kalimat). `Tokenizer.decode` tidak diubah.

## Artefak (di `/kaggle/working/outputs/metrics/`)

Semua artefak teks berformat **JSONL** (satu objek JSON per baris, ADR-011).

| File | Isi |
|---|---|
| `c15_kaggle_metrics.jsonl` | 20 record `epoch` + `final_eval` + `generation` |
| `c15_kaggle_generations.jsonl` | `sample` per prompt, `calibration`, `sampling_grid`, `meta` |
| `c15_kaggle_config.jsonl` | satu baris `meta` + satu baris `config` |
| `c15_kaggle_vocab.jsonl` | satu baris `{"token","id"}` per token |
| `minigpt_c15_kaggle_best.pt` | checkpoint `val_loss` per dokumen terendah |
| `minigpt_c15_kaggle_last.pt` | checkpoint epoch terakhir |

## Cara pakai

1. Upload `ayayo_dataset.zip` (dibangun ulang setelah data 1.800 artikel) sebagai Kaggle Dataset.
2. **Add Input** -> dataset tersebut. Path terdeteksi otomatis.
3. Settings -> Accelerator -> **GPU T4 x2**; internet tidak diperlukan.
4. **Run All**, lalu **Save Version** supaya `/kaggle/working` tersimpan.

> Hanya `cuda:0` yang dipakai. Model terlalu kecil untuk menutup biaya sinkronisasi
> `nn.DataParallel`, jadi GPU kedua idle dan notebook tidak mengklaim memakainya.

In [ ]:
# 1) Environment, path, device, seed
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

# --- Auto-discovery dataset ------------------------------------------------
# Kaggle me-mount dataset di /kaggle/input/<slug>. Nama slug tidak boleh
# di-hardcode karena berubah sesuai judul dataset. Cari folder yang berisi
# src/data/tokenizer.py, lalu pakai folder itu sebagai ASSET_ROOT.
INPUT_ROOT = Path("/kaggle/input")

def find_asset_root() -> Path:
    """Cari folder yang berisi `src/data/tokenizer.py`.

    Kaggle mengekstrak zip dataset, jadi isinya bisa di root mount
    (`/kaggle/input/<slug>/src/...`) atau terbungkus satu folder lagi.
    Pencarian dibatasi ke cwd + subfolder langsung `/kaggle/input` supaya
    tidak memindai direktori sistem Kaggle.
    """
    def has_src(p: Path) -> bool:
        return (p / "src" / "data" / "tokenizer.py").is_file()

    bases = [Path.cwd()]
    if INPUT_ROOT.is_dir():
        bases += sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir())

    for base in bases:                       # kasus normal: root langsung
        if has_src(base):
            return base
    for base in bases:                       # kasus terbungkus 1-3 folder
        for pattern in ("*", "*/*", "*/*/*"):
            for cand in sorted(base.glob(pattern)):
                if cand.is_dir() and has_src(cand):
                    return cand
    raise FileNotFoundError(
        "Tidak menemukan folder berisi src/data/tokenizer.py di "
        f"{[str(b) for b in bases]}. Pastikan dataset sudah di-Add Input."
    )

ASSET_ROOT = find_asset_root()
OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT_DIR = OUT_ROOT / "outputs" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# `from src.xxx import ...` butuh PARENT folder `src` ada di sys.path,
# bukan folder `src` itu sendiri.
if str(ASSET_ROOT) not in sys.path:
    sys.path.insert(0, str(ASSET_ROOT))

# --- Device ---------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True   # input shape tetap -> cudnn autotune aman

# --- Seed -----------------------------------------------------------------
SEED = 42

def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

print(f"[C15] ASSET_ROOT : {ASSET_ROOT}")
print(f"[C15] OUT_DIR    : {OUT_DIR}")
print(f"[C15] torch      : {torch.__version__}")
print(f"[C15] device     : {device}  ({gpu_name}, {n_gpu} GPU terlihat; dipakai cuda:0 saja)")

try:
    from src.data.tokenizer import Tokenizer
    from src.data.dataset import (TextDataset, create_dataloader,
                                  chunked_document_batches, iter_document_chunks)
    from src.model.minigpt import MiniGPT
    from src.config import Config, ModelConfig, TrainConfig
    from src.train import (set_seed, make_criterion, save_checkpoint,
                           write_metrics_jsonl, evaluate_per_document)
    from src.generate import (DecodeConfig, generate, generate_ids,
                              pick_sampling_temperature, tune_temperature)
    from src.postprocess import build_case_map, postprocess
    print("[C15] Imports OK")
except Exception as exc:  # pesan jelas, bukan traceback kabur
    raise RuntimeError(
        f"Gagal import dari {ASSET_ROOT}. "
        "Pastikan zip berisi folder 'src/' di root-nya."
    ) from exc


In [ ]:
# 2) Load data — train / val / test
def load_jsonl(path: Path) -> list:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

DATA_DIR = ASSET_ROOT / "data" / "processed"

def load_split(name: str):
    p = DATA_DIR / f"{name}.jsonl"
    if not p.is_file():
        return None
    return [d["content"] for d in load_jsonl(p)]

train_texts = load_split("train")
val_texts = load_split("val")
test_texts = load_split("test")  # opsional: hanya untuk evaluasi akhir

assert train_texts and val_texts, "train.jsonl / val.jsonl wajib ada di dataset"
print(f"[C15] train: {len(train_texts):>4} artikel, {sum(map(len, train_texts)):>9,} karakter")
print(f"[C15] val  : {len(val_texts):>4} artikel, {sum(map(len, val_texts)):>9,} karakter")
if test_texts:
    print(f"[C15] test : {len(test_texts):>4} artikel, {sum(map(len, test_texts)):>9,} karakter")
else:
    print("[C15] test : TIDAK ADA — evaluasi held-out akan dilewati (regenerate zip dengan test.jsonl)")


In [ ]:
# 3) Tokenizer + vocabulary (fit HANYA dari train -> tidak ada leakage)
import re

vocab_path = OUT_DIR / "c15_kaggle_vocab.jsonl"

# ADR-013: min_freq=2 membuang hapax. Pada C14, 8.173 dari 19.192 tipe (42,6%)
# muncul sekali, dan tiap tipe memakan 2 x d_model parameter (embed + head).
MIN_FREQ = 2
tok = Tokenizer()
tok.fit(train_texts, min_freq=MIN_FREQ)

def token_coverage(texts, tokenizer) -> float:
    """Proporsi token yang ADA di vocab (kebalikan dari UNK rate)."""
    total = hit = 0
    for t in texts:
        for token in re.findall(r"\w+|[^\w\s]", t.lower()):
            total += 1
            hit += token in tokenizer.token_to_id
    return hit / max(total, 1)

def unk_rate(texts) -> float:
    """Proporsi token val/test di luar vocab train (diagnostik OOV)."""
    total = hit = 0
    for t in texts:
        ids = tok.encode(t)
        total += len(ids)
        hit += sum(1 for i in ids if i == tok.special_tokens["<unk>"])
    return hit / max(total, 1)

val_unk = unk_rate(val_texts)
print(f"[C15] vocab_size : {tok.vocab_size:,}")
print(f"[C15] UNK rate   : val={val_unk:.3%}" + (f"  test={unk_rate(test_texts):.3%}" if test_texts else ""))
print(f"[C15] min_freq   : {MIN_FREQ}  (ADR-013)")
print(f"[C15] coverage   : train={token_coverage(train_texts, tok):.2%} "
      f"val={token_coverage(val_texts, tok):.2%}"
      + (f"  test={token_coverage(test_texts, tok):.2%}" if test_texts else ""))
print(f"[C15] log(V)     : {math.log(tok.vocab_size):.3f}  (val_loss di atas angka ini = lebih buruk dari tebakan uniform)")

# Vocab disimpan terpisah supaya checkpoint bisa dipakai ulang tanpa notebook ini.
# JSONL: satu entri {"token": ..., "id": ...} per baris, seragam dengan artefak lain.
with vocab_path.open("w", encoding="utf-8") as f:
    for token, token_id in sorted(tok.token_to_id.items(), key=lambda kv: kv[1]):
        f.write(json.dumps({"token": token, "id": token_id}, ensure_ascii=False) + "\n")
print(f"[C15] vocab      : {vocab_path}")

In [ ]:
# 4) Dataset + DataLoader + cek kontrak slicing (TECH_SPEC §4)
CONTEXT_LENGTH = 128   # ADR-015
BATCH_SIZE = 32
EPOCHS = 20            # ADR-014/015: 10 epoch belum plateau di C14
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

ds_train = TextDataset(train_texts, tok, context_length=CONTEXT_LENGTH)
ds_val = TextDataset(val_texts, tok, context_length=CONTEXT_LENGTH)
ds_test = TextDataset(test_texts, tok, context_length=CONTEXT_LENGTH) if test_texts else None

def make_loader(ds, shuffle: bool):
    loader = create_dataloader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                               pad_id=PAD_ID, context_length=CONTEXT_LENGTH)
    if shuffle:
        # DataLoader memakai `loader.generator` saat membuat iterator, jadi
        # menempelkannya setelah konstruksi membuat urutan shuffle reproducible.
        g = torch.Generator()
        g.manual_seed(SEED)
        loader.generator = g
    return loader

dl_train = make_loader(ds_train, shuffle=True)
dl_val = make_loader(ds_val, shuffle=False)
dl_test = make_loader(ds_test, shuffle=False) if ds_test is not None else None

print(f"[C15] Segmen : train={len(ds_train):,}  val={len(ds_val):,}" +
      (f"  test={len(ds_test):,}" if ds_test is not None else ""))
print(f"[C15] Batch  : train={len(dl_train):,}  val={len(dl_val):,}" +
      (f"  test={len(dl_test):,}" if dl_test is not None else ""))
print(f"[C15] Step   : {len(dl_train) * EPOCHS:,} optimizer step untuk {EPOCHS} epoch")

# --- Verifikasi bentuk tensor & invarian next-token -----------------------
xb, yb = next(iter(dl_val))
assert xb.shape == yb.shape == (min(BATCH_SIZE, len(ds_val)), CONTEXT_LENGTH), (xb.shape, yb.shape)
assert xb.dtype == torch.int64 and yb.dtype == torch.int64
assert xb.max().item() < tok.vocab_size

# x[i, t] harus sama dengan y[i, t-1] untuk semua posisi non-pad.
mask = yb[:, 1:] != PAD_ID
shift_ok = torch.equal(xb[:, 1:][mask], yb[:, :-1][mask])
assert shift_ok, "Invarian x[t] == y[t-1] gagal"

pad_frac = (yb == PAD_ID).float().mean().item()
print(f"[C15] Shape check : x={tuple(xb.shape)} y={tuple(yb.shape)} dtype={xb.dtype}")
print(f"[C15] Shift check : x[t] == y[t-1] untuk token non-pad -> {'OK' if shift_ok else 'GAGAL'}")
print(f"[C15] Padding val : {pad_frac:.2%}")

## Konfigurasi training C15

| Parameter | Nilai | Catatan |
|---|---|---|
| `n_layers / n_heads / d_model` | 4 / 4 / 64 | ADR-014 (depth naik setelah tying) |
| `context_length` | 128 | ADR-015 |
| `tie_embeddings` | True | ADR-014 |
| `min_freq` | 2 | ADR-013 |
| `dropout` | 0.2 | diwarisi dari C14 |
| `batch_size` | 32 | TECH_SPEC 6 |
| `epochs` | 20 | C14 belum plateau di epoch 10 |
| `lr` | 3e-4 -> 3e-5 | warmup 1% + cosine |
| `weight_decay` | 0.05 | diwarisi dari C14 |
| `grad_clip` | 1.0 | diwarisi dari C14 |
| `label_smoothing` | 0 | kontrak loss 5 tidak diubah |
| artifact | best + last | `best` dari `val_loss` per dokumen |

**Kenapa `n_layers` dinaikkan, bukan `d_model`.** Weight tying menghemat 1.228.288 parameter
(48% model C14). Parameter yang dihemat dipakai untuk menambah kedalaman, bukan lebar:
pada anggaran parameter yang sama, menambah layer memberi kapasitas koreksi bertahap yang
tidak bisa diberikan `d_model` lebih besar pada embedding yang sudah di-tie.

**Kenapa `min_freq=2`, bukan 3.** `min_freq=3` memangkas vocab ke 8.220 tipe dan menaikkan
UNK test ke >9%. `min_freq=2` memberi vocab 16.278 dengan coverage train 98,0% dan UNK test
4,66%. Karena `<unk>` mustahil diprediksi, menaikkan UNK akan menurunkan PPL secara semu
tanpa memperbaiki model; 2 adalah titik seimbang.

**Batasan yang harus ditulis jujur.** Penurunan perplexity akibat `min_freq=2` sebagian
adalah artefak metrik: `<unk>` menjadi token yang lebih sering dan lebih mudah diprediksi.
Laporan wajib mencantumkan coverage train 98,0%, UNK val/test, dan loss dengan token `<unk>`
dikecualikan agar perbandingan dengan C14 tetap adil.

In [ ]:
# 5) Config + model
N_LAYERS, N_HEADS, D_MODEL, D_FF = 4, 4, 64, 256   # ADR-014: depth 2 -> 4
TIE_EMBEDDINGS = True                                           # ADR-014
DROPOUT = 0.2
LR_PEAK, LR_MIN = 3e-4, 3e-5
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
WARMUP_FRAC = 0.01

cfg = Config(
    model=ModelConfig(
        vocab_size=tok.vocab_size, n_layers=N_LAYERS, n_heads=N_HEADS,
        d_model=D_MODEL, context_length=CONTEXT_LENGTH, d_ff=D_FF,
        dropout=DROPOUT, layer_norm_eps=1e-5,
        tie_embeddings=TIE_EMBEDDINGS, min_freq=MIN_FREQ,
    ),
    train=TrainConfig(
        seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS,
        lr=LR_PEAK, weight_decay=WEIGHT_DECAY,
        pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID,
    ),
)

set_seed(SEED)

model = MiniGPT(
    vocab_size=tok.vocab_size, n_layers=N_LAYERS, n_heads=N_HEADS,
    d_model=D_MODEL, context_length=CONTEXT_LENGTH, dropout=DROPOUT,
    layer_norm_eps=1e-5, tie_embeddings=TIE_EMBEDDINGS,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
# Dengan tying, lm_head.weight IS token_emb.weight, jadi hitung sekali.
n_embed_params = model.token_emb.weight.numel()
n_head_params = 0 if TIE_EMBEDDINGS else model.lm_head.weight.numel()
n_emb_total = n_embed_params + n_head_params
n_tokens_train = sum(len(tok.encode(t)) for t in train_texts)
print(f"[C15] Params         : {n_params:,}")
print(f"[C15]   embedding    : {n_embed_params:,} ({n_embed_params / n_params:.1%})")
print(f"[C15]   LM head      : {n_head_params:,} ({'di-tie ke embedding - ADR-014' if TIE_EMBEDDINGS else 'terpisah'})")
print(f"[C15]   blok+norm+PE : {n_params - n_emb_total:,}")
print(f"[C15] Token train    : {n_tokens_train:,} -> {n_params / max(n_tokens_train,1):.2f} param/token")
print(f"[C15] tie_embeddings : {TIE_EMBEDDINGS} (shared weight = {model.lm_head.weight is model.token_emb.weight})")

# Forward pass uji: bentuk logits harus [B, T, V]
with torch.no_grad():
    probe = model(xb.to(device))
assert probe.shape == (xb.size(0), CONTEXT_LENGTH, tok.vocab_size), probe.shape
assert probe.dtype == torch.float32
print(f"[C15] Forward OK    : logits {tuple(probe.shape)} float32")
del probe
if device.type == "cuda":
    torch.cuda.empty_cache()

# Simpan konfigurasi + metadata environment untuk reproducibility laporan
run_meta = {
    "experiment": "C15-kaggle",
    "config": cfg.to_dict(),
    "training_overrides_vs_baseline": {
        "dropout": "0.1 -> 0.2",
        "weight_decay": "0.01 -> 0.05",
        "lr_schedule": "constant 3e-4 -> warmup 1% + cosine 3e-4..3e-5",
        "grad_clip": "none -> 1.0",
        "checkpoint": "last epoch -> best val_loss (+ last)",
        "min_freq": f"1 -> {MIN_FREQ} (ADR-013)",
        "tie_embeddings": f"False -> {TIE_EMBEDDINGS} (ADR-014)",
        "n_layers": "2 -> 4 (ADR-014)",
        "context_length": "64 -> 128 (ADR-015)",
        "subset": "1000 -> 1800 artikel (ADR-015)",
        "epochs": "10 -> 20",
    },
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "context_length": CONTEXT_LENGTH,
    "vocab_size": tok.vocab_size,
    "n_params": n_params,
    "n_params_embedding": n_embed_params,
    "n_params_lm_head": n_head_params,
    "n_tokens_train": n_tokens_train,
    "params_per_token": round(n_params / max(n_tokens_train, 1), 4),
    "tie_embeddings": TIE_EMBEDDINGS,
    "min_freq": MIN_FREQ,
    "train_token_coverage": round(token_coverage(train_texts, tok), 6),
    "val_unk_rate": round(val_unk, 6),
    "test_unk_rate": round(unk_rate(test_texts), 6) if test_texts else None,
    "articles": {"train": len(train_texts), "val": len(val_texts),
                 "test": len(test_texts) if test_texts else 0},
    "segments": {"train": len(ds_train), "val": len(ds_val),
                 "test": len(ds_test) if ds_test is not None else 0},
    "device": str(device),
    "gpu_name": gpu_name,
    "gpu_count_visible": n_gpu,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "asset_root": str(ASSET_ROOT),
    "started_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
config_path = OUT_DIR / "c15_kaggle_config.jsonl"
# JSONL: satu baris meta + satu baris konfigurasi, seragam dengan artefak lain.
with config_path.open("w", encoding="utf-8") as f:
    f.write(json.dumps({"event": "meta", **{k: v for k, v in run_meta.items()
                                            if k != "config"}},
                       ensure_ascii=False) + "\n")
    f.write(json.dumps({"event": "config", **run_meta["config"]},
                       ensure_ascii=False) + "\n")
print(f"[C15] Config disimpan: {config_path}")

In [ ]:
# 6) Evaluator: token-weighted (training) + dokumen-level (pelaporan)
import statistics

def evaluate_token_weighted(model, loader, criterion, device):
    """Loss rata-rata per TOKEN valid (bukan per batch)."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                       # [B, T, V]
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            n_tok = int((y != PAD_ID).sum().item())
            total_loss += float(loss.item()) * n_tok
            total_tokens += n_tok
    return total_loss / max(total_tokens, 1), total_tokens


def evaluate_documents(model, texts, tokenizer, context_length=CONTEXT_LENGTH,
                       batch_size=BATCH_SIZE, pad_id=PAD_ID, ignore_unk=True):
    """Loss per DOKUMEN (window non-overlap) + CI 95%.

    `evaluate_token_weighted` di atas menilai setiap window stride-1 dari
    dokumen yang sama, sehingga satu token dihitung berkali-kali dan angkanya
    terlihat jauh lebih pasti daripada buktinya. Di sini tiap dokumen jadi satu
    sampel dan lebar CI melaporkan ketidakpastian sebenarnya.

    `ignore_unk=True` mengecualikan target `<unk>` dari rata-rata. ADR-013
    mewajibkan ini: token di luar vocab mustahil diprediksi, jadi memasukkannya
    berarti mencampur coverage tokenizer ke dalam angka kualitas model.
    """
    batches = chunked_document_batches(
        texts, tokenizer, context_length=context_length, batch_size=batch_size,
        pad_id=pad_id,
    )
    ignore = (UNK_ID,) if ignore_unk else ()
    mean, sd, n_docs = evaluate_per_document(model, batches, pad_id=pad_id,
                                             ignore_ids=ignore)
    half = 1.96 * sd / math.sqrt(n_docs) if n_docs > 1 else float("nan")
    return {"loss": mean, "ppl": ppl(mean), "sd": sd, "n_docs": n_docs,
            "ci95": half, "loss_lo": mean - half, "loss_hi": mean + half,
            "ignore_unk": ignore_unk}


def ppl(loss: float) -> float:
    return float(math.exp(min(loss, 20.0)))


def fmt_ci(stat) -> str:
    return f"{stat['loss']:.4f} +/-{stat['ci95']:.4f} (PPL {stat['ppl']:.0f}, n={stat['n_docs']})"


optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR_PEAK, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)
)
criterion = make_criterion(pad_id=PAD_ID)

total_steps = len(dl_train) * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_FRAC))

def lr_lambda(step: int) -> float:
    """Warmup linear 1% lalu cosine dari LR_PEAK turun ke LR_MIN."""
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return (LR_MIN + (LR_PEAK - LR_MIN) * cosine) / LR_PEAK

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"[C15] Optimizer   : AdamW(lr={LR_PEAK:g}->{LR_MIN:g}, wd={WEIGHT_DECAY})")
print(f"[C15] Schedule    : warmup {warmup_steps:,} step + cosine, total {total_steps:,} step")
print(f"[C15] Criterion   : CrossEntropyLoss(ignore_index={PAD_ID})")
print("[C15] Val protocol: per dokumen (window non-overlap) + CI 95%")

In [ ]:
# 7) Training - EPOCHS (=20) epoch, metrik + checkpoint per epoch
metrics_path = OUT_DIR / "c15_kaggle_metrics.jsonl"
ckpt_last = OUT_DIR / "minigpt_c15_kaggle_last.pt"
ckpt_best = OUT_DIR / "minigpt_c15_kaggle_best.pt"

history = []
best_val = float("inf")          # doc-level val loss = kriteria pemilihan
best_epoch = 0
global_step = 0

def save_best():
    save_checkpoint(str(ckpt_best), model, vocab=tok.token_to_id, config={
        **cfg.to_dict(),
        "checkpoint": {"epoch": best_epoch, "val_loss_doc": best_val,
                       "selection": "min val_loss per dokumen"},
    })

t0_total = time.time()
for epoch in range(1, EPOCHS + 1):
    t0_ep = time.time()
    model.train()
    sum_loss, sum_tokens, sum_gradnorm = 0.0, 0, 0.0

    for x, y in dl_train:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)                                   # [B, T, V]
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        global_step += 1

        n_tok = int((y != PAD_ID).sum().item())
        sum_loss += float(loss.item()) * n_tok
        sum_tokens += n_tok
        sum_gradnorm += float(grad_norm)

    train_loss = sum_loss / max(sum_tokens, 1)
    # kriteria pemilihan: loss per dokumen, bukan per window redundant
    val_doc = evaluate_documents(model, val_texts, tok, ignore_unk=True)
    val_doc_raw = evaluate_documents(model, val_texts, tok, ignore_unk=False)
    val_loss = val_doc["loss"]
    val_tok_weighted, val_tokens = evaluate_token_weighted(model, dl_val, criterion, device)

    elapsed = time.time() - t0_ep
    cur_lr = scheduler.get_last_lr()[0]
    improved = val_loss < best_val

    rec = {
        "event": "epoch",
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "val_loss": round(val_loss, 6),
        "val_loss_ci95": round(val_doc["ci95"], 6),
        "val_loss_with_unk": round(val_doc_raw["loss"], 6),
        "val_loss_tokenweighted": round(val_tok_weighted, 6),
        "train_ppl": round(ppl(train_loss), 3),
        "val_ppl": round(ppl(val_loss), 3),
        "gap": round(val_loss - train_loss, 6),
        "lr": round(cur_lr, 8),
        "grad_norm_mean": round(sum_gradnorm / max(1, len(dl_train)), 4),
        "epoch_sec": round(elapsed, 2),
        "tokens_per_sec": round(sum_tokens / max(elapsed, 1e-9)),
    }
    history.append(rec)

    if improved:
        best_val, best_epoch = val_loss, epoch

    write_metrics_jsonl(history, str(metrics_path))
    save_checkpoint(str(ckpt_last), model, vocab=tok.token_to_id, config=cfg.to_dict())
    if improved:
        save_best()

    print(f"[C15] Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} (ppl {rec['train_ppl']:.1f}) "
          f"val={val_loss:.4f}+/-{val_doc['ci95']:.3f} (ppl {rec['val_ppl']:.1f}) "
          f"gap={rec['gap']:+.3f} lr={cur_lr:.2e} | {elapsed:.1f}s"
          + ("  <- best" if improved else ""))

total_time = time.time() - t0_total

# Epoch yang statistiknya seri dengan yang terbaik: memilih satu di antaranya
# tidak dapat dijustifikasi oleh data.
tied = [r for r in history if r["val_loss"] <= best_val + r["val_loss_ci95"]]
tied_epochs = [r["epoch"] for r in tied]

print()
print(f"[C15] Total training : {total_time / 60:.1f} menit ({total_time:.0f}s)")
print(f"[C15] Best epoch     : {best_epoch} (val_loss={best_val:.4f}, val_ppl={ppl(best_val):.1f})")
print(f"[C15] Seri dalam CI  : epoch {tied_epochs} (selisih < CI 95%)")
print(f"[C15] Checkpoint best: {ckpt_best}")
print(f"[C15] Checkpoint last: {ckpt_last}")

In [ ]:
# 8) Evaluasi akhir dari checkpoint BEST: dokumen-level + CI 95%
best_ckpt = torch.load(ckpt_best, map_location="cpu", weights_only=False)
# from_config menyaring kunci yang bukan parameter konstruktor (d_ff, min_freq)
# dan tetap memakai tie_embeddings dari checkpoint. Membangun ulang model secara
# manual seperti sebelumnya akan MENGHILANGKAN weight tying (model jadi punya
# dua matriks terpisah dengan nilai sama) sehingga jumlah parameter berbeda dari
# yang dilaporkan saat training.
model_best = MiniGPT.from_config(best_ckpt["config"]["model"]).to(device)
model_best.load_state_dict(best_ckpt["state_dict"])
assert model_best.tie_embeddings == TIE_EMBEDDINGS, "tie_embeddings hilang saat restore"
model_best.eval()
print(f"[C15] Loaded best checkpoint: epoch={best_ckpt['config']['checkpoint']['epoch']}")

val_stat = evaluate_documents(model_best, val_texts, tok, ignore_unk=True)
val_stat_raw = evaluate_documents(model_best, val_texts, tok, ignore_unk=False)
last_rec = history[-1]
final = {
    "event": "final_eval",
    "best_epoch": best_epoch,
    "tied_epochs": tied_epochs,
    "val_loss": round(val_stat["loss"], 6),
    "val_loss_ci95": round(val_stat["ci95"], 6),
    "val_ppl": round(val_stat["ppl"], 3),
    "val_docs": val_stat["n_docs"],
    "val_loss_with_unk": round(val_stat_raw["loss"], 6),
    "eval_ignores_unk": True,
    "train_ppl_at_best_epoch": history[best_epoch - 1]["train_ppl"],
    "train_ppl_at_last_epoch": last_rec["train_ppl"],
    "total_seconds": round(total_time, 2),
    "device": str(device),
    "gpu_name": gpu_name,
    "torch_version": torch.__version__,
    "vocab_size": tok.vocab_size,
    "n_params": n_params,
    "eval_protocol": "per-document, non-overlapping windows, <unk> targets excluded (ADR-013), CI95 from 1.96*sd/sqrt(n_docs)",
}

if test_texts:
    test_stat = evaluate_documents(model_best, test_texts, tok, ignore_unk=True)
    test_stat_raw = evaluate_documents(model_best, test_texts, tok, ignore_unk=False)
    final.update({
        "test_loss": round(test_stat["loss"], 6),
        "test_loss_ci95": round(test_stat["ci95"], 6),
        "test_ppl": round(test_stat["ppl"], 3),
        "test_docs": test_stat["n_docs"],
        "test_loss_with_unk": round(test_stat_raw["loss"], 6),
    })
else:
    print("[C15] test.jsonl tidak ada -> test_loss/test_ppl dilewati")

with open(metrics_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(final) + "\n")

print()
print("=== Evaluasi akhir (checkpoint best, per dokumen) ===")
print(f"  val  {fmt_ci(val_stat)}  [termasuk unk: {val_stat_raw['loss']:.4f}]")
if "test_loss" in final:
    print(f"  test {fmt_ci(test_stat)}  [termasuk unk: {test_stat_raw['loss']:.4f}]")
print(f"  acuan uniform: PPL >= {tok.vocab_size:,} (loss >= {math.log(tok.vocab_size):.2f})")
print(f"  best epoch={best_epoch}; seri dalam CI: {tied_epochs}")
print(f"  gap val-train di epoch terakhir = {last_rec['gap']:+.3f} nat")

In [ ]:
# 9) Decoding: kalibrasi temperature, generasi, dan metrik kualitas
# Semua knob di sini hanya mengubah PEMILIHAN token berikutnya. Bobot model,
# arsitektur, dan kontrak `decode` (TECH_SPEC 3) tidak disentuh.

import re

# --- 9a. Prompt held-out: 5 dari val + 5 dari test (total 10) -------------
def prompt_from(texts, start, length=6):
    toks = re.findall(r"\w+|[^\w\s]", texts[start].lower())
    return " ".join(toks[:length])

def pick_prompts(sources, count, length=6, seen=None):
    """Ambil `count` prompt unik dari beberapa daftar teks (gabungan).

    `seen` dibagi antar-sumber supaya prompt yang sama tidak muncul dua kali
    ketika val dan test kebetulan berisi artikel serupa.
    """
    seen = seen if seen is not None else set()
    picks = []
    for texts in sources:
        for i in range(len(texts)):
            prompt = prompt_from(texts, i, length)
            key = prompt
            if key in seen:
                continue
            seen.add(key)
            picks.append(prompt)
            if len(picks) == count:
                return picks
    return picks

PROMPTS = pick_prompts([val_texts, test_texts or []], 10)
assert len(PROMPTS) == 10, f"butuh 10 prompt held-out, dapat {len(PROMPTS)}"
print("[C15] 10 prompt held-out (val/test, bukan train):")
for prompt in PROMPTS:
    print(f"  - {prompt}")

GEN_SEED = 1234
MAX_NEW_TOKENS = 80

# --- 9b. Kalibrasi temperature (teacher forcing pada val) ----------------
# NLL/ECE mengukur kualitas probabilitas. NLL minimum BUKAN rekomendasi
# sampling: pada model kecil yang overfit, NLL terus turun sampai T>1.5 padahal
# teksnya sudah acak. Nilai NLL dipakai untuk laporan; T untuk sampling dipilih
# di 9c dengan batasan koherensi.
calib_batches = list(chunked_document_batches(
    val_texts, tok, context_length=CONTEXT_LENGTH, batch_size=BATCH_SIZE, pad_id=PAD_ID,
))
CALIB_T, calib_rows, T_AT_BOUNDARY = tune_temperature(
    model_best, [(x, y) for x, y, _ in calib_batches],
    grid=(0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5, 2.0),
)
if T_AT_BOUNDARY:
    print(f"[C15] PERINGATAN: T NLL-optimal di batas grid ({CALIB_T}); perluas grid.")
print("\n[C15] Kalibrasi (NLL per token pada val; makin kecil makin terkalibrasi):")
for t, nll, acc, ece in calib_rows:
    print(f"  T={t:<4} NLL={nll:.4f} PPL={math.exp(nll):8.1f} acc={acc:.3f} ECE={ece:.4f}")

# --- 9c. Temperature sampling: batasi ke yang koheren -------------------
SAMPLE_T, sample_rows = pick_sampling_temperature(
    model_best, tok, PROMPTS, grid=(0.8, 0.9, 1.0, 1.1, 1.2, 1.3),
    max_new_tokens=MAX_NEW_TOKENS, calibration=calib_rows,
)
print("\n[C15] Uji kelayakan temperature sampling (10 prompt held-out):")
for t, d1, dup in sample_rows:
    ok = "OK" if dup == 0 and d1 >= 0.75 else "tolak"
    print(f"  T={t:<4} distinct-1={d1:.3f} dup-trigram={dup:<3} {ok}")
print(f"[C15] T sampling dipakai: {SAMPLE_T} "
      f"(NLL-optimal={CALIB_T} hanya untuk pelaporan kalibrasi)")

GREEDY_CFG = DecodeConfig(do_sample=False, no_repeat_ngram_size=3, ban_unk=True,
                          repetition_penalty=1.0)
SAMPLE_CFG = DecodeConfig(do_sample=True, temperature=SAMPLE_T, top_p=0.9, top_k=0,
                          repetition_penalty=1.1, no_repeat_ngram_size=3,
                          ban_unk=True, seed=GEN_SEED)

case_map = build_case_map(train_texts, min_count=3)
print(f"[C15] case_map   : {len(case_map):,} entri proper-noun dari train")

# --- 9d. Metrik kualitas generasi ---------------------------------------
# `no_repeat_ngram_size=3` MELARANG trigram berulang, jadi "dup-trigram = 0"
# adalah hasil pemaksaan, bukan bukti model membaik. Yang informatif adalah
# 4-gram: di luar cakupan larangan, jadi kalau berulang itu keputusan model.
TRAIN_STREAM = " " + " ".join(
    " ".join(re.findall(r"\w+|[^\w\s]", text.lower())) for text in train_texts
) + " "

def longest_train_span(tokens, cap=16):
    """Span terpanjang yang identik dengan teks train (indikator hafalan)."""
    for n in range(min(len(tokens), cap), 3, -1):
        for start in range(len(tokens) - n + 1):
            if (" " + " ".join(tokens[start:start + n]) + " ") in TRAIN_STREAM:
                return n, " ".join(tokens[start:start + n])
    return 0, ""

def generation_metrics(untrimmed, tokens):
    n = max(len(tokens), 1)
    trigrams = [tuple(tokens[i:i + 3]) for i in range(len(tokens) - 2)]
    quads = [tuple(tokens[i:i + 4]) for i in range(len(tokens) - 3)]
    span_len, span_text = longest_train_span(tokens)
    # Diukur SEBELUM trim: kalau diukur sesudah, angkanya selalu "selesai"
    # karena trim sendiri yang memotong di titik. Yang benar: apakah model
    # berhenti di batas kalimat tanpa bantuan kita.
    terminates = untrimmed.rstrip().endswith((".", "!", "?"))
    return {
        "distinct1": round(len(set(tokens)) / n, 4),
        "dup_trigram": len(trigrams) - len(set(trigrams)),
        "dup_4gram": len(quads) - len(set(quads)),
        "ends_at_sentence_pre_trim": bool(terminates),
        "verbatim_span_tokens": span_len,
        "verbatim_span_text": span_text,
    }

def render(gen_ids, prompt):
    """Token baru -> teks siap laporan (detokenisasi, kapitalisasi, trim)."""
    return postprocess(gen_ids, tok.id_to_token, prompt=prompt,
                       case_map=case_map, trim=True)

def render_raw(gen_ids, prompt):
    """Sama tanpa trim, untuk mengukur apakah model berhenti di batas kalimat."""
    return postprocess(gen_ids, tok.id_to_token, prompt=prompt,
                       case_map=case_map, trim=False)

def run_generation(cfg, label):
    rows = []
    print(f"\n=== {label} ===")
    for prompt in PROMPTS:
        gen = generate_ids(model_best, tok, prompt=prompt,
                           max_new_tokens=MAX_NEW_TOKENS,
                           context_length=CONTEXT_LENGTH, bos_id=BOS_ID,
                           eos_id=EOS_ID, cfg=cfg)
        text = render(gen.ids, prompt)
        untrimmed = render_raw(gen.ids, prompt)
        tokens = re.findall(r"\w+|[^\w\s]", untrimmed.lower())
        metrics = generation_metrics(untrimmed, tokens)
        row = {
            "event": "sample",
            "mode": label,
            "prompt": prompt,
            "output": text,
            "output_untrimmed": untrimmed,
            "new_tokens": len(gen.ids),
            "stopped_on_eos": gen.stopped_on_eos,
            "hit_token_cap": len(gen.ids) >= MAX_NEW_TOKENS,
            **metrics,
            "config": {k: v for k, v in vars(cfg).items()},
        }
        rows.append(row)
        print(f"\n> {prompt}")
        print(f"  {text}")
        print(f"  [token={row['new_tokens']} eos={row['stopped_on_eos']} "
              f"dup4={row['dup_4gram']} verbatim={row['verbatim_span_tokens']} "
              f"selesai={row['ends_at_sentence_pre_trim']}]")
    return rows

generations = run_generation(GREEDY_CFG, "greedy+no_repeat_3") \
            + run_generation(SAMPLE_CFG, f"sampling T={SAMPLE_T} top_p=0.9 rep=1.1 ng=3")

# --- 9e. Simpan artefak (JSONL: satu objek JSON per baris) ---------------
gen_path = OUT_DIR / "c15_kaggle_generations.jsonl"
with gen_path.open("w", encoding="utf-8") as f:
    for row in [{"event": "calibration", "T": t, "nll": nll, "acc": acc, "ece": ece}
                for t, nll, acc, ece in calib_rows]:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    for t, d1, dup in sample_rows:
        f.write(json.dumps({"event": "sampling_grid", "T": t,
                            "distinct1": d1, "dup_trigram": dup},
                           ensure_ascii=False) + "\n")
    f.write(json.dumps({"event": "meta", "checkpoint": str(ckpt_best),
                        "best_epoch": best_epoch, "max_new_tokens": MAX_NEW_TOKENS,
                        "nll_optimal_temperature": CALIB_T,
                        "at_grid_boundary": T_AT_BOUNDARY,
                        "sampling_temperature": SAMPLE_T,
                        "decode_note": "postprocess.py: detokenize + restore_case "
                                       "+ trim_to_sentence; unk dibanned"},
                       ensure_ascii=False) + "\n")
    for row in generations:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Ringkas ke metrics JSONL (satu baris, ADR-011)
with metrics_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps({
        "event": "generation",
        "checkpoint": str(ckpt_best),
        "best_epoch": best_epoch,
        "val_loss": round(val_stat["loss"], 6),
        "sampling_temperature": SAMPLE_T,
        "nll_optimal_temperature": CALIB_T,
        "n_prompts": len(PROMPTS),
        "samples": generations,
        "total_seconds": round(total_time, 2),
        "device": str(device),
        "gpu_name": gpu_name,
        "torch_version": torch.__version__,
        "vocab_size": tok.vocab_size,
        "n_params": n_params,
    }, ensure_ascii=False) + "\n")

print(f"\n[C15] Generations: {gen_path}")
print(f"[C15] Metrics    : {metrics_path}")

In [ ]:
# 10) Verifikasi output + tabel ringkas (tanpa pandas, baca JSONL per baris)
rows = [json.loads(line) for line in metrics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
epoch_rows = [r for r in rows if r.get("event") == "epoch"]
assert len(epoch_rows) == EPOCHS, f"harus ada {EPOCHS} record epoch, dapat {len(epoch_rows)}"

header = (f"{'ep':>3} | {'train':>8} | {'val':>8} | {'+/-CI':>6} | {'tr_ppl':>8} | "
          f"{'va_ppl':>8} | {'gap':>7} | {'lr':>9} | {'sec':>6}")
print("=== Training history (val = loss per dokumen) ===")
print(header)
print("-" * len(header))
for r in epoch_rows:
    mark = " *" if r["epoch"] == best_epoch else (" ~" if r["epoch"] in tied_epochs else "  ")
    print(f"{r['epoch']:>3} | {r['train_loss']:>8.4f} | {r['val_loss']:>8.4f} | "
          f"{r['val_loss_ci95']:>6.3f} | {r['train_ppl']:>8.1f} | "
          f"{r['val_ppl']:>8.1f} | {r['gap']:>+7.3f} | {r['lr']:>9.2e} | "
          f"{r['epoch_sec']:>6.1f}{mark}")
print("* = checkpoint best dipakai; ~ = seri statistik dengan best (selisih < CI)")

first, last = epoch_rows[0], epoch_rows[-1]
if len(epoch_rows) < 2:
    trend = "tidak dapat dinilai (hanya 1 epoch)"
elif abs(last["val_loss"] - first["val_loss"]) <= last["val_loss_ci95"]:
    trend = "STABIL (selisih di bawah CI, tidak berbeda nyata)"
elif last["val_loss"] < first["val_loss"]:
    trend = "TURUN"
else:
    trend = "NAIK"
print(f"\nTren val_loss epoch 1 -> {len(epoch_rows)}: {first['val_loss']:.4f} -> "
      f"{last['val_loss']:.4f} ({trend})")
print(f"Gap val-train epoch terakhir = {last['gap']:+.3f} nat")
print(f"Val loss: per-dokumen {last['val_loss']:.4f} (tanpa <unk>) vs "
      f"{last['val_loss_with_unk']:.4f} (dengan <unk>) vs "
      f"{last['val_loss_tokenweighted']:.4f} (token-weighted, stride-1)")
print(f"  selisih dengan-unk - tanpa-unk = {last['val_loss_with_unk'] - last['val_loss']:+.4f} nat "
      "(kontribusi token OOV, bukan kualitas model)")

for r in rows:
    if r.get("event") == "final_eval":
        print(f"\n[final_eval] val {r['val_loss']:.4f} +/-{r['val_loss_ci95']:.4f} "
              f"(PPL {r['val_ppl']:.0f}, n={r['val_docs']} dokumen, tanpa <unk>; "
              f"dengan <unk> {r['val_loss_with_unk']:.4f})"
              + (f", test {r['test_loss']:.4f} +/-{r['test_loss_ci95']:.4f} "
                 f"(PPL {r['test_ppl']:.0f}, n={r['test_docs']})" if "test_ppl" in r else "")
              + f" @ best epoch {r['best_epoch']} (seri: {r['tied_epochs']})")

# --- Verifikasi format artefak (semua teks JSONL) ------------------------
ARTIFACT_JSONL = ("c15_kaggle_metrics.jsonl", "c15_kaggle_generations.jsonl",
                  "c15_kaggle_config.jsonl", "c15_kaggle_vocab.jsonl")
for name in ARTIFACT_JSONL:
    path = OUT_DIR / name
    assert path.is_file(), f"artefak hilang: {name}"
    lines = [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert lines, f"{name} kosong"
    for line in lines:
        json.loads(line)          # setiap baris harus JSON valid
    print(f"  OK  {name:<34} {len(lines):>6} baris JSONL valid")
stray = sorted(p.name for p in OUT_DIR.glob("c15_kaggle_*") if p.suffix != ".jsonl")
assert not stray, f"artefak non-JSONL tersisa: {stray}"

# --- Ringkasan kualitas generasi -----------------------------------------
# Angka diambil dari file generasi JSONL, bukan dihitung ulang dari teks yang
# sudah di-trim. Kuncinya:
#  - dup_4gram: di luar cakupan larangan ng3, jadi berulang = keputusan model.
#    dup_trigram SELALU 0 karena dilarang decoder, bukan karena model bagus.
#  - ends_at_sentence_pre_trim: diukur sebelum trim, kalau sesudah selalu 100%.
#  - verbatim_span_tokens: span identik dengan train; naik = makin menghafal.
import re

gen_rows = [json.loads(line) for line in gen_path.read_text(encoding="utf-8").splitlines()
            if line.strip()]
samples = [r for r in gen_rows if r.get("event") == "sample"]
if samples:
    by_mode = {}
    for s in samples:
        by_mode.setdefault(s["mode"], []).append(s)

    print("\n=== Ringkasan generasi (10 prompt held-out) ===")
    header2 = (f"{'mode':<32} | {'selesai*':>8} | {'dup4':>5} | {'distinct1':>9} | "
               f"{'verbatim':>8}")
    print(header2)
    print("-" * len(header2))
    for mode, items in by_mode.items():
        n = len(items)
        ended = sum(1 for s in items if s["ends_at_sentence_pre_trim"]) / n
        dup4 = sum(s["dup_4gram"] for s in items) / n
        dist1 = sum(s["distinct1"] for s in items) / n
        span = sum(s["verbatim_span_tokens"] for s in items) / n
        print(f"{mode:<32} | {ended:>7.0%} | {dup4:>5.2f} | {dist1:>9.3f} | {span:>8.1f}")
    print("* selesai = output mentah berakhir di . ! ? TANPA trim (diukur pre-trim)")
    print("  dup4 = 4-gram berulang (ng3 tidak melarang 4-gram, jadi ini jujur)")
    print("  verbatim = rata-rata span terpanjang yang identik dengan train (token)")

    worst = max(samples, key=lambda s: s["verbatim_span_tokens"])
    if worst["verbatim_span_tokens"] >= 10:
        print(f"\n  PERINGATAN hafalan: {worst['verbatim_span_tokens']} token identik train")
        print(f"    {worst['verbatim_span_text']}")

print("\n=== Artefak (semua teks JSONL) ===")
for p in sorted(OUT_DIR.glob("c15_kaggle_*")) + sorted(OUT_DIR.glob("minigpt_c15_kaggle_*")):
    n_lines = ""
    if p.suffix == ".jsonl":
        n_lines = f"{sum(1 for _ in p.open(encoding='utf-8')):>7} baris  "
    print(f"  {p.stat().st_size:>12,} B  {n_lines}{p.name}")

## Setelah Run All -> Save Version

1. **Save Version** (Save & Run All) supaya isi `/kaggle/working/outputs/metrics/` ikut
   tersimpan di tab *Output* notebook version.
2. File yang perlu diunduh untuk laporan (semua teks berformat JSONL):
   - `c15_kaggle_metrics.jsonl` — bukti 20 epoch + evaluasi akhir + generasi (ADR-011)
   - `c15_kaggle_generations.jsonl` — 20 sampel + grid kalibrasi + metrik hafalan
   - `c15_kaggle_config.jsonl` — konfigurasi + metadata environment
   - `c15_kaggle_vocab.jsonl` — vocabulary terpakai
   - `minigpt_c15_kaggle_best.pt` — checkpoint model terbaik
3. Angka yang wajib masuk laporan:
   - `train_loss` / `val_loss` / perplexity per epoch (tabel dari sel terakhir)
   - `val_loss_ci95` dan `tied_epochs` (epoch mana yang seri secara statistik)
   - epoch terbaik + alasan pemilihan checkpoint
   - `gap` val-train di epoch akhir
   - `val_loss` dan `test_loss` per dokumen pada checkpoint best, beserta CI 95%
   - contoh generasi beserta 10 prompt held-out-nya
   - `dup_4gram`, `ends_at_sentence_pre_trim`, dan `verbatim_span_tokens` per mode
     (lihat tabel di sel pertama untuk cara membacanya)
   - parameter total, `params_per_token`, coverage vocab, dan UNK rate per split

### Cara membaca hasil
- Acuan acak untuk C15 adalah `vocab_size` 16.278 (`log(V) = 9,70`). `ppl` di atas angka
  itu berarti belum lebih baik dari tebakan uniform.
- Bandingkan **selalu dengan CI 95%**. Dua epoch yang selisihnya di bawah CI adalah seri;
  checkpoint "best" di antara mereka adalah pilihan arbitrer dan harus ditulis begitu.
- Angka C15 **tidak bisa** dibandingkan langsung dengan C14: vocabulary berbeda (16.278 vs
  19.192) dan token `<unk>` lebih sering. Perplexity antarrun dengan vocab berbeda bukan
  besaran yang sama. Bandingkan `test_loss` setelah mengecualikan token `<unk>`, atau
  bandingkan kualitas generasi secara kualitatif.
- `val_loss` per dokumen (n=159) adalah estimasi atas 159 artikel; lebar CI mencerminkan
  itu. `test_loss` (n=200) adalah angka untuk klaim generalisasi.
- Model 1,24 juta parameter di 1.440 artikel tetap tidak akan menghasilkan berita faktual.
  Yang realistis: meniru *gaya* berita (kapitalisasi nama tempat, pola kutipan
  `"..., kata dia"`, angka/tanggal) dengan 1-2 kalimat lokal yang utuh.
- Gap `val - train` yang melebar tiap epoch = overfitting. Pada C15 knob kontrak
  (`min_freq`, tying, `context_length`) sudah dipakai; langkah berikutnya adalah data
  tambahan atau `label_smoothing` (§5, butuh persetujuan).

### Batasan yang harus ditulis jujur
- GPU: hanya `cuda:0`; GPU kedua T4x2 idle karena model terlalu kecil untuk 2 GPU.
- Val = 10% dari train (ADR-012), jadi `val_ppl` bukan ukuran generalisasi penuh;
  `test_ppl` (200 artikel held-out) yang dipakai untuk klaim generalisasi.
- `min_freq=2` menaikkan UNK val/test. Sebagian penurunan perplexity adalah artefak
  metrik, bukan pemahaman bahasa. Wajib dilaporkan bersama coverage 98,0%.
- Hasil generasi adalah demonstrasi pendidikan, bukan sumber berita yang akurat.
- C15 mengubah kontrak yang dilindungi baseline lewat ADR-013/014/015. Semua perubahan
  tercatat dan dapat dipertanggungjawabkan; arsitektur inti (decoder-only, causal MHA
  manual, sinusoidal PE, pre-LayerNorm) tidak berubah.